In [3]:
%env ANYWIDGET_HMR=1
from guidepost import Guidepost, Campsite

# gp = Guidepost()
cs = Campsite()

env: ANYWIDGET_HMR=1


In [4]:
cs.test_server()

0


In [5]:
import pandas as pd
# jobs_data = pd.read_parquet("../data/test_data_med.parquet")
# jobs_data = pd.read_parquet("../data/kestrel_data_2024_01_28_subsample.parquet")
# jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-THETA_20240101_20241231.csv.gz", compression='gzip')
jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')
# df = pd.read_csv("../data/ANL-ALCF-MACHINESTATUS-POLARIS_20250101_20251231.csv.gz", compression='gzip')

# jobs_data
# gp.records = jobs_data
cs.records = jobs_data

/tmp/ipykernel_83028/2327723536.py:5: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')


In [4]:
jobs_data
# jobs_data['Y'] = (jobs_data["EXIT_STATUS"] == 0).astype(int)
# jobs_data['LOCATION'].unique()

,JOB_NAME,COBALT_JOBID,MACHINE_NAME,QUEUED_TIMESTAMP,QUEUED_DATE_ID,START_TIMESTAMP,START_DATE_ID,END_TIMESTAMP,END_DATE_ID,USERNAME_GENID,...,OVERBURN_CORE_HOURS,IS_OVERBURN,GPUS_REQUESTED,IS_PYTHON,PYTHON_EXECUTABLE_PATH,PYTHON_EXECUTABLE_VERSION,TASK_EXIT_CODES,IS_TASK_EXIT_CODES_NON_ZERO,SCIENCE_FIELD,SCIENCE_FIELD_SHORT
0,3123554.polaris,0,polaris,2024-12-31 23:12:02,20241231,2024-12-31 23:12:08,20241231,2025-01-01 00:02:41,20250101,45432069360014,...,0.0,0,8,0,NaN,NaN,NaN,-1,Chemistry:Quantum Chemistry,Chemistry
1,3123553.polaris,0,polaris,2024-12-31 23:04:21,20241231,2024-12-31 23:04:27,20241231,2025-01-01 00:05:03,20250101,56895850159645,...,0.0,0,4,0,NaN,NaN,NaN,-1,Chemistry:General,Chemistry
2,3123562.polaris,0,polaris,2024-12-31 23:30:04,20241231,2024-12-31 23:30:11,20241231,2025-01-01 00:05:42,20250101,45840552982799,...,0.0,0,4,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
3,3123555.polaris,0,polaris,2024-12-31 23:17:20,20241231,2024-12-31 23:17:26,20241231,2025-01-01 00:18:04,20250101,52819878632584,...,0.0,0,4,0,NaN,NaN,NaN,-1,Computer Science,Computer Science
4,3123541.polaris,0,polaris,2024-12-31 22:28:09,20241231,2024-12-31 22:44:29,20241231,2025-01-01 00:33:58,20250101,13652914070362,...,0.0,0,40,0,NaN,NaN,NaN,-1,Computer Science,Computer Science
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262695,6829301.polaris,0,polaris,2025-12-31 23:26:02,20251231,2025-12-31 23:26:05,20251231,2025-12-31 23:30:02,20251231,6625408822030,...,0.0,0,-1,0,NaN,NaN,NaN,-1,Computer Science,Computer Science
262696,6829248.polaris,0,polaris,2025-12-31 21:46:05,20251231,2025-12-31 22:39:11,20251231,2025-12-31 23:32:39,20251231,27137196728557,...,0.0,0,-1,0,NaN,NaN,NaN,-1,Engineering:Aerodynamics,Engineering
262697,6829291.polaris,0,polaris,2025-12-31 23:11:20,20251231,2025-12-31 23:11:24,20251231,2025-12-31 23:46:24,20251231,10144454012692,...,0.0,0,-1,0,NaN,NaN,NaN,-1,Computer Science,Computer Science
262698,6829283.polaris,0,polaris,2025-12-31 22:45:47,20251231,2025-12-31 22:45:51,20251231,2025-12-31 23:46:37,20251231,33458719883793,...,0.0,0,-1,0,NaN,NaN,NaN,-1,Chemistry:Biochemistry,Chemistry


In [5]:
cs

In [13]:
!pip install "vegafusion>=2.0.3"
!pip install "vl-convert-python>=1.8.0"
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

alt.data_transformers.enable("vegafusion")

def hypothesis_over_request_vs_used_viz(
    df: pd.DataFrame,
    requested_col: str = "REQUESTED_CORE_HOURS",
    used_col: str = "USED_CORE_HOURS",
    bins: int = 100
) -> alt.Chart:
    """
    Build a single Altair chart visualizing the primary quantity of interest
    for the per-job over-requesting hypothesis:
      H0: mean_diff = 0 vs H1: mean_diff > 0
      where diff = REQUESTED_CORE_HOURS - USED_CORE_HOURS.

    Primary analysis rules (per the clarifications):
      - Exclude records with REQUESTED_CORE_HOURS <= 0 from the primary analysis.
      - Compute per-job diffs = REQUESTED_CORE_HOURS - USED_CORE_HOURS.
      - Provide a histogram of diffs, with:
          - a vertical line at the mean_diff (red)
          - a vertical line at 0 (blue, dashed) as the H0 reference
      - Title includes: n, mean_diff, 95% CI, and one-sided p-value for H0: μ_diff=0 vs μ_diff>0
    Returns:
      An Altair Chart (vega-lite spec) that can be rendered in Jupyter, JupyterLab, or exported.
    """
    # Basic cleaning
    df_clean = df.dropna(subset=[requested_col, used_col])
    df_primary = df_clean[df_clean[requested_col] > 0].copy()

    if df_primary.empty:
        raise ValueError("No records with positive REQUESTED_CORE_HOURS for primary analysis.")

    # Primary quantity: per-job difference
    df_primary["diff"] = df_primary[requested_col] - df_primary[used_col]

    # Summary statistics for primary hypothesis test
    n = int(df_primary.shape[0])
    mean_diff = float(df_primary["diff"].mean())
    std_diff = float(df_primary["diff"].std(ddof=1))

    # Standard error and CI for mean difference (two-sided CI; used for annotation)
    se = std_diff / np.sqrt(n) if n > 0 else np.nan
    if n > 1 and std_diff > 0:
        t_crit = float(stats.t.ppf(0.975, df=n - 1))
        ci_low = mean_diff - t_crit * se
        ci_high = mean_diff + t_crit * se
        t_stat = mean_diff / se
        p_value = float(1 - stats.t.cdf(t_stat, df=n - 1))  # one-sided p-value
    else:
        ci_low = ci_high = np.nan
        p_value = np.nan

    title_text = (
        f"Over-requesting (per-job): mean_diff = {mean_diff:.2f} hours "
        f"(requested - used), n={n}; 95% CI [{ci_low:.2f}, {ci_high:.2f}]. "
        f"One-sided p-value (H0: μ_diff=0 vs H1: μ_diff>0) = {p_value:.3g}"
    )

    # Histogram of diffs
    diff_df = df_primary[["diff"]].copy()
    hist = (
        alt.Chart(diff_df)
        .mark_bar(opacity=0.8, color="#4C78A8")
        .encode(
            x=alt.X("diff:Q", bin=alt.Bin(maxbins=bins),
                    title="REQUESTED_CORE_HOURS − USED_CORE_HOURS (hours)"),
            y=alt.Y("count()", title="Number of jobs")
        )
    )

    # Vertical lines for reference thresholds
    mean_line = (
        alt.Chart(pd.DataFrame({"mean_diff": [mean_diff]}))
        .mark_rule(color="red", thickness=2)
        .encode(x="mean_diff:Q")
    )

    zero_line = (
        alt.Chart(pd.DataFrame({"zero": [0]}))
        .mark_rule(color="blue", strokeDash=[6, 4])
        .encode(x="zero:Q")
    )

    chart = (hist + mean_line + zero_line).properties(
        width=700,
        height=350,
        title=title_text
    )

    return chart

chart = hypothesis_over_request_vs_used_viz(jobs_data)
chart.display()  # in some environments
# chart.show()     # in some environments
# chart.to_html("over_request_viz.html")  # export if needed

alt.LayerChart(...)

In [1]:
import pandas as pd
import numpy as np
import altair as alt
import scipy.stats as stats

# Statsmodels imports (for mixed-effects modeling)
import statsmodels.formula.api as smf
import patsy

def delta_core_covariate_adjusted_visualization(
    data,
    exit_col='EXIT_STATUS',
    req_col='REQUESTED_CORE_HOURS',
    used_col='USED_CORE_HOURS',
    user_col='USERNAME_GENID',
    sci_field_col='SCIENCE_FIELD_SHORT',
    queue_col='QUEUE_NAME',
    location_col='LOCATION',
    start_date_col='START_DATE_ID',
    delta_out_col='Delta_core'
):
    """
    Build a covariate-adjusted visualization for the primary Delta_core metric.

    Data assumptions:
    - data is a pandas DataFrame containing:
      - EXIT_STATUS (int): primary filter; we use EXIT_STATUS == 0 for the primary analysis
      - REQUESTED_CORE_HOURS (float)
      - USED_CORE_HOURS (float)
      - USERNAME_GENID (categorical/ordinal) for random intercept
      - SCIENCE_FIELD_SHORT (categorical)
      - QUEUE_NAME (categorical)
      - LOCATION (categorical)
      - START_DATE_ID (numeric, can be int64)
    - If any of these covariates are missing, they should be handled upstream or via simple imputation.

    Returns:
    - An Altair Chart object (single visualization) that shows:
      - covariate-adjusted mean Δ_core for a typical profile
      - 95% CI interval
      - a p-value annotation for H0: μΔ = 0 (one-sided Ha: μΔ > 0)
      - a visual reference line at Δ_core = 0
    """
    # 1) Prepare data: compute Delta_core and filter to EXIT_STATUS == 0
    df = data.copy()

    # Ensure necessary columns exist
    required_cols = [
        exit_col, req_col, used_col, user_col,
        sci_field_col, queue_col, location_col, start_date_col
    ]
    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    df[delta_out_col] = df[req_col] - df[used_col]

    # Primary analysis: only successful jobs
    df_valid = df[(df[exit_col] == 0)].copy()
    df_valid = df_valid.dropna(subset=[delta_out_col, start_date_col,
                                       sci_field_col, queue_col, location_col, start_date_col, user_col])

    if df_valid.shape[0] == 0:
        raise ValueError("No valid rows after filtering for EXIT_STATUS == 0 and complete covariates.")

    # 2) Fit a mixed-effects model: Δ_core ~ C(SCIENCE_FIELD_SHORT) + C(QUEUE_NAME) + C(LOCATION) + START_DATE_ID
    #    with random intercept by USERNAME_GENID
    formula = (
        f"{delta_out_col} ~ C({sci_field_col}) + C({queue_col}) + C({location_col}) + {start_date_col}"
    )

    # Ensure proper dtypes for categorical predictors
    df_valid[sci_field_col] = df_valid[sci_field_col].astype('category')
    df_valid[queue_col] = df_valid[queue_col].astype('category')
    df_valid[location_col] = df_valid[location_col].astype('category')
    # START_DATE_ID can remain numeric (int/float)

    # Try a mixed-effects model first
    model = None
    res = None
    try:
        model = smf.mixedlm(formula, df_valid, groups=df_valid[user_col])
        res = model.fit(reml=False)
    except Exception as e:
        # Fallback to fixed-effects model (no random effects)
        try:
            model = smf.ols(formula, df_valid)
            res = model.fit()
        except Exception as e2:
            raise RuntimeError(f"Model fitting failed: {e} | Fallback failed: {e2}")

    # 3) Build a typical covariate profile for marginal (population) mean
    #    - Use the most frequent category for each categorical covariate
    typical = pd.DataFrame({
        sci_field_col: [df_valid[sci_field_col].mode()[0]],
        queue_col: [df_valid[queue_col].mode()[0]],
        location_col: [df_valid[location_col].mode()[0]],
        start_date_col: [df_valid[start_date_col].mean()],
        delta_out_col: [0.0]  # placeholder for y; not used in exog construction
    })

    # 4) Build the fixed-effects design matrix for the typical profile using Patsy
    #    We need only the fixed-effects part; use the same formula to ensure compatibility
    try:
        y_typ, X_typ = patsy.dmatrices(formula, typical, return_type='dataframe')
    except Exception as e:
        raise RuntimeError(f"Failed to construct design matrix for typical profile: {e}")

    # 5) Compute covariate-adjusted mean and SE from fixed-effects
    #    fe_params and cov_params come from the fitted results
    fe_params = res.fe_params  # fixed-effects parameters
    cov_fe = res.cov_params()    # covariance of fixed effects

    # Align X_typ shape with fe_params
    # Some ranks may differ if Patsy added/dropped terms; ensure same columns
    # We'll reindex X_typ to match fe_params.index
    try:
        X_typ_aligned = X_typ.reindex(columns=fe_params.index, fill_value=0.0)
    except Exception:
        # Fallback: try to convert to numpy arrays with aligned shapes
        X_typ_aligned = X_typ.values  # risky, but best effort

    # Compute mean_hat = exog * betas
    mean_hat = float(np.dot(X_typ_aligned.values, fe_params.values))

    # Compute SE from fixed-effects covariance
    try:
        var_fixed = np.dot(X_typ_aligned.values, np.dot(cov_fe.values, X_typ_aligned.values.T))
        se = float(np.sqrt(var_fixed[0]))
    except Exception:
        # If alignment fails, fall back to a large SE to avoid misinterpretation
        se = float(np.nan)

    # If SE is not computable, raise a warning-like value
    if not np.isfinite(se) or se <= 0:
        # Provide a fallback: use the standard error from the model's residuals (rough fallback)
        se = float(np.sqrt(max(res.scale, 1e-6)))  # not exact for fixed-effects, but avoids NaN

    # Compute p-value for H0: μΔ = 0, one-sided Ha: μΔ > 0
    z = mean_hat / se
    p_value_one_sided = float(1 - stats.norm.cdf(z))

    # 6) Build the Altair visualization data
    viz_df = pd.DataFrame({
        'label': ['Covariate-adjusted mean Δ_core (EXIT_STATUS = 0)'],
        'mean': [mean_hat],
        'low': [mean_hat - 1.96 * se],
        'high': [mean_hat + 1.96 * se],
        'p_value': [p_value_one_sided],
        'p_text': [f"p = {p_value_one_sided:.3f} (one-sided)"],
    })

    # 7) Create Altair chart: point + 95% CI + reference line at 0 + p-value annotation
    base = alt.Chart(viz_df).mark_text(align='left', baseline='middle').encode(
        x=alt.datum.mean, y=alt.datum.label
    )

    point = alt.Chart(viz_df).mark_point(size=120, color='steelblue').encode(
        x='mean:Q',
        y=alt.Y('label:N', sort=None)
    )

    ci = alt.Chart(viz_df).mark_errorbar(color='steelblue').encode(
        x='low:Q',
        x2='high:Q',
        y=alt.Y('label:N', sort=None)
    )

    # Horizontal axis reference: a vertical line at x=0 (Δ_core = 0)
    # We add a simple vertical line as an additional layer.
    zero_line = alt.Chart(pd.DataFrame({'x': [0], 'label': viz_df['label'][0]})).mark_rule(color='red', strokeDash=[4, 4]).encode(
        x='x:Q',
        y=alt.Y('label:N', sort=None)
    )

    # Optional: text annotation of the p-value near the point
    p_text = alt.Chart(viz_df).mark_text(align='left', baseline='bottom', dx=5, color='black').encode(
        x='mean:Q',
        y=alt.Y('label:N', sort=None),
        text='p_text:N'
    )

    chart = (ci + point + zero_line + p_text).properties(
        width=700,
        height=120,
        title='Covariate-adjusted mean Δ_core (EXIT_STATUS = 0) with 95% CI'
    )

    return chart

chart = delta_core_covariate_adjusted_visualization( jobs_data, exit_col='EXIT_STATUS', req_col='REQUESTED_CORE_HOURS', used_col='USED_CORE_HOURS', user_col='USERNAME_GENID', sci_field_col='SCIENCE_FIELD_SHORT', queue_col='QUEUE_NAME', location_col='LOCATION', start_date_col='START_DATE_ID' )

chart # In a notebook, this renders the Altair chart

chart = hypothesis_over_request_vs_used_viz(jobs_data)
chart.display()  # in some environments
chart.show()     # in some environments
chart.to_html("over_request_viz.html")

NameError: name 'jobs_data' is not defined

In [ ]:
!pip install scipy
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

def visualize_over_request_hypothesis(
    df,
    req_col='REQUESTED_CORE_HOURS',
    used_cols=None,
    delta=1e-6,
    winsor_pct=0.01,
    date_col='START_TIMESTAMP',
    include_primary_filter=True,
    date_col_format='datetime',
    min_obs=30,
    random_seed=0
):
    """
    Create a single visualization to test the primary hypothesis:
      E[W_i] > 0 where W_i = log( REQUEST_i / ( USED_i + δ ) ),
    for per-user-month observations.

    Parameters
    - df: pandas DataFrame containing the data.
    - req_col: name of the requested core hours column (default 'REQUESTED_CORE_HOURS').
    - used_cols: list or tuple of column names to compute USED_TOTAL.
                 If None, attempts to use USED_CORE_HOURS or CAPABILITY & NONCAPABILITY sums.
    - delta: small constant added to denominator to avoid log(0).
    - winsor_pct: fraction for winsorizing REQUEST and USED totals (e.g., 0.01 for 1st/99th percentiles).
    - date_col: name of date column to derive monthly trend inset (optional).
    - include_primary_filter: if True, apply END_STATUS=0 and EXIT_CODE=0 filter when columns exist.
    - date_col_format: 'datetime' or 'string'; if 'datetime', will parse to datetime.
    - min_obs: minimum number of observations required to produce the visualization.
    - random_seed: seed for any randomness (if sampling is used).
    
    Returns
    - fig: matplotlib Figure object containing the visualization.
    - df_out: a DataFrame after processing, containing computed columns like USED_TOTAL, W, O, etc.
    - stats: dict with mean(W), 95% CI, p-value (one-sided), exp(mean(W)) as interpretability, N, and O_i prevalence.
    """

    # 1) Determine USED_TOTAL
    if used_cols is None:
        if {'CAPABILITY_USAGE_CORE_HOURS', 'NONCAPABILITY_USAGE_CORE_HOURS'}.issubset(df.columns):
            df = df.copy()
            df['USED_TOTAL'] = df['CAPABILITY_USAGE_CORE_HOURS'] + df['NONCAPABILITY_USAGE_CORE_HOURS']
        elif {'USED_CORE_HOURS'}.issubset(df.columns):
            df = df.copy()
            df['USED_TOTAL'] = df['USED_CORE_HOURS']
        else:
            raise ValueError("Could not determine USED_TOTAL: supply used_cols or ensure USED_CORE_HOURS exists.")
    else:
        # sum the provided used_cols
        df = df.copy()
        if len(used_cols) == 1:
            df['USED_TOTAL'] = df[used_cols[0]]
        elif len(used_cols) >= 2:
            df['USED_TOTAL'] = df[used_cols[0]] + df[used_cols[1]]
        else:
            raise ValueError("used_cols must have at least 1 element.")

    # 2) Basic cleaning
    df['REQUEST'] = df[req_col]

    # Optional: apply primary inclusion filter based on END_STATUS/EXIT_CODE
    if include_primary_filter:
        mask = pd.Series([True] * len(df), index=df.index)
        for c in ['END_STATUS', 'EXIT_CODE']:
            if c in df.columns:
                mask &= (df[c] == 0)
        df = df[mask]

    # Remove missing values in key columns
    df = df.dropna(subset=['REQUEST', 'USED_TOTAL']).copy()

    # Clamp negatives to missing (primary analysis assumption)
    df.loc[df['REQUEST'] < 0, 'REQUEST'] = np.nan
    df.loc[df['USED_TOTAL'] < 0, 'USED_TOTAL'] = np.nan
    df = df.dropna(subset=['REQUEST', 'USED_TOTAL']).copy()

    if len(df) < min_obs:
        raise ValueError(f"Not enough observations after cleaning. n={len(df)}, min_obs={min_obs}")

    # 3) Winsorize REQUEST and USED_TOTAL
    if winsor_pct > 0:
        req_lo = df['REQUEST'].quantile(winsor_pct)
        req_hi = df['REQUEST'].quantile(1 - winsor_pct)
        used_lo = df['USED_TOTAL'].quantile(winsor_pct)
        used_hi = df['USED_TOTAL'].quantile(1 - winsor_pct)

        df['REQUEST_W'] = df['REQUEST'].clip(req_lo, req_hi)
        df['USED_W'] = df['USED_TOTAL'].clip(used_lo, used_hi)
    else:
        df['REQUEST_W'] = df['REQUEST']
        df['USED_W'] = df['USED_TOTAL']

    # 4) Compute W_i and O_i
    df['W'] = np.log((df['REQUEST_W'] + 0.0) / (df['USED_W'] + delta))

    # Over-requesting indicators
    df['REL_EXCESS'] = (df['REQUEST_W'] - df['USED_W']) / (df['USED_W'] + delta)
    df['ABS_DIFF'] = df['REQUEST_W'] - df['USED_W']
    df['O'] = np.where(
        ((df['REL_EXCESS'] >= 0.25) & (df['ABS_DIFF'] >= 1.0)) |
        ((df['USED_W'] == 0) & (df['REQUEST_W'] > 0)),
        1, 0
    )

    # 5) Compute statistics for the primary test
    W = df['W'].values
    n = len(W)
    mean_W = float(np.mean(W))
    se_W = float(np.std(W, ddof=1) / np.sqrt(n))
    df_t = n - 1
    t_stat = mean_W / se_W if se_W > 0 else np.nan
    p_value = float(1 - stats.t.cdf(t_stat, df_t)) if not np.isnan(t_stat) else np.nan
    tcrit = stats.t.ppf(0.975, df_t)
    ci_low = mean_W - tcrit * se_W
    ci_high = mean_W + tcrit * se_W
    ratio = float(np.exp(mean_W))

    # 6) Prepare the plot (single figure with KDE of W, plus an inset if date info exists)
    sns.set(style='whitegrid')
    fig, ax = plt.subplots(figsize=(8, 5))

    # Density of W
    sns.kdeplot(W, ax=ax, fill=True, color='steelblue', alpha=0.6, bw_adjust=0.8)

    # Reference lines
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='H0: E[W]=0')
    ax.axvline(mean_W, color='green', linestyle='-', linewidth=2, label='Sample mean W')

    # Labels and title
    ax.set_xlabel('W = log( REQUEST / USED + δ )')
    ax.set_ylabel('Density')
    ax.set_title('Distribution of W_i (per-user-month) and primary test result')

    # Summary text box
    textstr = (
        f"n = {n}\n"
        f"mean(W) = {mean_W:.4f}\n"
        f"95% CI       [{ci_low:.4f}, {ci_high:.4f}]\n"
        f"p-value (one-sided) = {p_value:.3g}\n"
        f"exp(mean(W)) = {ratio:.3f}\n"
        f"O_i prevalence = {df['O'].mean():.3f}"
    )
    props = dict(boxstyle='round', facecolor='white', alpha=0.9)
    ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right', bbox=props)

    # Optional inset: monthly mean W if a date column is present
    inset_ax = None
    if date_col is not None and date_col in df.columns:
        # Try to parse date column
        try:
            if date_col_format == 'datetime':
                df['DATE'] = pd.to_datetime(df[date_col], errors='coerce')
            else:
                df['DATE'] = pd.to_datetime(df[date_col].astype(str), errors='coerce')
        except Exception:
            df['DATE'] = pd.to_datetime(df[date_col], errors='coerce')

        if df['DATE'].notnull().any():
            df['MONTH'] = df['DATE'].dt.to_period('M')
            monthly = df.groupby('MONTH', observed=False)['W'].mean().reset_index()
            if len(monthly) > 1:
                inset_ax = ax.inset_axes([0.55, 0.18, 0.38, 0.25])
                sns.barplot(x=monthly['MONTH'].astype(str),
                            y=monthly['W'],
                            ax=inset_ax,
                            color='gray')
                inset_ax.set_ylabel('Mean W')
                inset_ax.set_xlabel('Month')
                inset_ax.set_title('Monthly mean W')
                inset_ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    stats_out = {
        'mean_W': mean_W,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_value_one_sided': p_value,
        'exp_mean_W': ratio,
        'n': int(n),
        'O_rate': float(df['O'].mean())
    }

    return fig, df, stats_out

fig, df_out, stats = visualize_over_request_hypothesis(jobs_data)
fig.show()


In [ ]:
jobs_data.columns

In [ ]:
gp.vis_configs = {
        'x': 'START_TIMESTAMP',
        'y': 'USED_CORE_HOURS',
        'color': 'processors_req',
        'color_agg': 'avg',
        'categorical': 'user',
        'facet_by': 'MACHINE_PARTITION'
}

In [ ]:
gp

In [ ]:
I am interested in relationships between memory efficiency, power, and job type.

In [ ]:
gp.selection

In [ ]:
import pandas as pd

# Split data into long-running and shorter jobs

long = jobs_data[jobs_data['wallclock_req_seconds'] > 14400]

short = jobs_data[jobs_data['wallclock_req_seconds'] <= 14400]

# Compute average CPU efficiency for each group

avg_cpu_eff_long = long['cpu_eff'].mean()

avg_cpu_eff_short = short['cpu_eff'].mean()

# Hypothesis: avg CPU efficiency for long < avg CPU efficiency for short

result = avg_cpu_eff_long < avg_cpu_eff_short

In [ ]:
print(result, avg_cpu_eff_long, avg_cpu_eff_short)

In [ ]:
import pandas as pd

def evaluate(df):

    avg_standard = df.loc[df['partition'] == 'standard', 'wallclock_req_seconds'].mean()
    
    avg_short = df.loc[df['partition'] == 'short', 'wallclock_req_seconds'].mean()

    print(avg_standard, avg_short)
    
    return avg_standard > avg_short

evaluate(jobs_data)

In [ ]:
import pandas as pd

def evaluate(df):

    corr = df['cpu_eff'].corr(df['wallclock_req_seconds'])

    result = corr < -0.5

    print(corr)

    return result

In [ ]:
evaluate_hypothesis(jobs_data)

In [ ]:
import pandas as pd

def evaluate(df):

    # Long-running jobs (wallclock_req_seconds > 3600)
    
    long_running = df[df['wallclock_req_seconds'] > 3600]
    
    avg_long = long_running['cpu_eff'].mean()
    
    # Average CPU efficiency across all jobs
    
    avg_all = df['cpu_eff'].mean()

    print(avg_long, avg_all)
    
    return avg_long > avg_all

In [ ]:
def evaluate_hypothesis(df):

    # Compute failure rate for account 'sipv'

    total_sipv = df[df['account'] == 'sipv'].shape[0]

    failed_sipv = df[(df['account'] == 'sipv') & (df['state'] == 'FAILED')].shape[0]

    failure_rate_sipv = failed_sipv / total_sipv if total_sipv > 0 else float('nan')

    # Compute failure rate for all other accounts

    total_others = df[df['account'] != 'sipv'].shape[0]

    failed_others = df[(df['account'] != 'sipv') & (df['state'] == 'FAILED')].shape[0]

    failure_rate_others = failed_others / total_others if total_others > 0 else float('nan')

    # If we don't have data for either group, return False (cannot conclude higher rate)

    if total_sipv == 0 or total_others == 0:

        return False

    print(failure_rate_sipv, failure_rate_others)

    return bool(failure_rate_sipv > failure_rate_others)